In [15]:
from pathlib import Path

base_path = Path("sort_object_tracking")
input_video_path = base_path / "output_video"

input_video_path.mkdir(parents=True, exist_ok=True)

In [5]:
%%writefile sort_object_tracking/utils.py

import numpy as np

def iou(bb_test, bb_gt):
    """
    Compute IoU between two boxes.
    Boxes in format [x1, y1, x2, y2]
    """
    # Find intersection rectangle
    xx1 = np.maximum(bb_test[0], bb_gt[0])
    yy1 = np.maximum(bb_test[1], bb_gt[1])
    xx2 = np.minimum(bb_test[2], bb_gt[2])
    yy2 = np.minimum(bb_test[3], bb_gt[3])

    # Intersection area
    w = np.maximum(0., xx2 - xx1)
    h = np.maximum(0., yy2 - yy1)
    intersection = w * h

    # Union area
    area_test = ((bb_test[2] - bb_test[0]) *
                 (bb_test[3] - bb_test[1]))
    area_gt   = ((bb_gt[2]   - bb_gt[0])   *
                 (bb_gt[3]   - bb_gt[1]))
    union = area_test + area_gt - intersection

    return intersection / union


def convert_bbox_to_z(bbox):
    """
    Convert [x1,y1,x2,y2] to Kalman state [u,v,s,r]

    u = center x
    v = center y
    s = scale (area)
    r = aspect ratio (w/h)
    """
    w = bbox[2] - bbox[0]   # width
    h = bbox[3] - bbox[1]   # height
    u = bbox[0] + w / 2     # center x
    v = bbox[1] + h / 2     # center y
    s = w * h                # area
    r = w / float(h)         # aspect ratio

    return np.array([u, v, s, r]).reshape((4, 1))


def convert_x_to_bbox(x, score=None):
    """
    Convert Kalman state [u,v,s,r] back to [x1,y1,x2,y2]
    """
    w = np.sqrt(x[2] * x[3])      # width  = sqrt(s × r)
    h = x[2] / w                   # height = s / w

    box = [
        x[0] - w / 2,   # x1
        x[1] - h / 2,   # y1
        x[0] + w / 2,   # x2
        x[1] + h / 2    # y2
    ]

    if score is None:
        return np.array(box).reshape((1, 4))
    else:
        return np.array(box + [score]).reshape((1, 5))


def associate_detections_to_trackers(detections, trackers, iou_threshold=0.3):
    """
    Hungarian algorithm assignment.

    detections:  (N, 5) array [x1,y1,x2,y2,score]
    trackers:    (M, 5) array [x1,y1,x2,y2,track_id]

    Returns:
        matches:         matched (detection_idx, tracker_idx) pairs
        unmatched_dets:  detections with no match
        unmatched_trks:  trackers with no match
    """
    from scipy.optimize import linear_sum_assignment

    if len(trackers) == 0:
        return (
            np.empty((0, 2), dtype=int),
            np.arange(len(detections)),
            np.empty((0,), dtype=int)
        )

    # Build IoU matrix (cost matrix)
    iou_matrix = np.zeros((len(detections), len(trackers)), dtype=np.float32)

    for d, det in enumerate(detections):
        for t, trk in enumerate(trackers):
            iou_matrix[d, t] = iou(det, trk)

    # Hungarian algorithm — maximize IoU (minimize negative IoU)
    row_ind, col_ind = linear_sum_assignment(-iou_matrix)
    matched_indices = np.stack([row_ind, col_ind], axis=1)

    # Filter out matches below IoU threshold
    unmatched_dets = []
    for d in range(len(detections)):
        if d not in matched_indices[:, 0]:
            unmatched_dets.append(d)

    unmatched_trks = []
    for t in range(len(trackers)):
        if t not in matched_indices[:, 1]:
            unmatched_trks.append(t)

    matches = []
    for m in matched_indices:
        if iou_matrix[m[0], m[1]] < iou_threshold:
            # IoU too low → treat as unmatched
            unmatched_dets.append(m[0])
            unmatched_trks.append(m[1])
        else:
            matches.append(m.reshape(1, 2))

    if len(matches) == 0:
        matches = np.empty((0, 2), dtype=int)
    else:
        matches = np.concatenate(matches, axis=0)

    return (
        matches,
        np.array(unmatched_dets),
        np.array(unmatched_trks)
    )

Writing sort_object_tracking/utils.py


In [7]:
%%writefile sort_object_tracking/sort.py
import numpy as np
from filterpy.kalman import KalmanFilter
from utils import (convert_bbox_to_z,
                   convert_x_to_bbox,
                   associate_detections_to_trackers)


class KalmanBoxTracker:
    """
    Represents one tracked object using a Kalman Filter.
    State: [u, v, s, r, u_dot, v_dot, s_dot]
    """
    count = 0  # class variable — global ID counter

    def __init__(self, bbox):
        """
        bbox: [x1, y1, x2, y2] initial detection
        """
        # ── Kalman Filter Setup ──────────────────────────
        # dim_x = 7 (state size)
        # dim_z = 4 (measurement size: u,v,s,r)
        self.kf = KalmanFilter(dim_x=7, dim_z=4)

        # State transition matrix F
        self.kf.F = np.array([
            [1,0,0,0,1,0,0],   # u  = u  + u_dot
            [0,1,0,0,0,1,0],   # v  = v  + v_dot
            [0,0,1,0,0,0,1],   # s  = s  + s_dot
            [0,0,0,1,0,0,0],   # r  = r  (constant)
            [0,0,0,0,1,0,0],   # u_dot constant
            [0,0,0,0,0,1,0],   # v_dot constant
            [0,0,0,0,0,0,1],   # s_dot constant
        ], dtype=float)

        # Measurement matrix H
        # We observe u,v,s,r but NOT velocities
        self.kf.H = np.array([
            [1,0,0,0,0,0,0],
            [0,1,0,0,0,0,0],
            [0,0,1,0,0,0,0],
            [0,0,0,1,0,0,0],
        ], dtype=float)

        # Measurement noise R
        # How much do we trust the detector?
        self.kf.R[2:, 2:] *= 10.  # less trust in scale/ratio

        # Initial uncertainty P
        # High uncertainty for velocities (we don't know them yet)
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P        *= 10.

        # Process noise Q
        # How much can state change unexpectedly?
        self.kf.Q[-1, -1] *= 0.01   # s_dot changes slowly
        self.kf.Q[4:, 4:] *= 0.01   # velocities change slowly

        # Initialize state with first detection
        self.kf.x[:4] = convert_bbox_to_z(bbox)

        # Track metadata
        self.time_since_update = 0    # frames since last matched detection
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.history = []             # predicted box history
        self.hits = 0                 # total times matched
        self.hit_streak = 0           # consecutive matches
        self.age = 0                  # total frames alive

    def update(self, bbox):
        """
        Called when this track is matched to a detection.
        Runs Kalman UPDATE step.
        """
        self.time_since_update = 0
        self.history = []
        self.hits += 1
        self.hit_streak += 1
        self.kf.update(convert_bbox_to_z(bbox))

    def predict(self):
        """
        Runs Kalman PREDICT step.
        Called every frame before matching.
        """
        # Prevent negative scale
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0

        self.kf.predict()
        self.age += 1

        if self.time_since_update > 0:
            self.hit_streak = 0    # streak broken if not matched

        self.time_since_update += 1
        self.history.append(convert_x_to_bbox(self.kf.x))

        return self.history[-1]    # return predicted bbox

    def get_state(self):
        """Return current state as [x1,y1,x2,y2]"""
        return convert_x_to_bbox(self.kf.x)


class SORT:
    """
    Main SORT tracker.
    Manages all KalmanBoxTrackers.
    """
    def __init__(self, max_age=1, min_hits=3, iou_threshold=0.3):
        """
        max_age:       delete track after this many unmatched frames
        min_hits:      confirm track after this many consecutive matches
        iou_threshold: minimum IoU for valid match
        """
        self.max_age       = max_age
        self.min_hits      = min_hits
        self.iou_threshold = iou_threshold
        self.trackers      = []       # list of KalmanBoxTrackers
        self.frame_count   = 0

    def update(self, detections):
        """
        Main update function. Called every frame.

        detections: numpy array (N, 5) [x1,y1,x2,y2,score]
                    or (N, 4) [x1,y1,x2,y2]

        Returns: numpy array (M, 5) [x1,y1,x2,y2,track_id]
        """
        self.frame_count += 1

        # ── Step 1: Predict new positions for all trackers ──
        predicted_boxes = np.zeros((len(self.trackers), 5))
        to_delete = []

        for t, tracker in enumerate(self.trackers):
            pred = tracker.predict()[0]   # [x1,y1,x2,y2]
            predicted_boxes[t] = [*pred, 0]

            # Mark tracker if prediction is invalid (NaN)
            if np.any(np.isnan(pred)):
                to_delete.append(t)

        # Remove invalid trackers
        predicted_boxes = np.ma.compress_rows(
            np.ma.masked_invalid(predicted_boxes)
        )
        for t in reversed(to_delete):
            self.trackers.pop(t)

        # ── Step 2: Match predictions to detections ──
        matched, unmatched_dets, unmatched_trks = \
            associate_detections_to_trackers(
                detections,
                predicted_boxes,
                self.iou_threshold
            )

        # ── Step 3: Update matched trackers ──
        for m in matched:
            self.trackers[m[1]].update(detections[m[0]])

        # ── Step 4: Create new trackers for unmatched detections ──
        for i in unmatched_dets:
            new_tracker = KalmanBoxTracker(detections[i])
            self.trackers.append(new_tracker)

        # ── Step 5: Build output + delete old trackers ──
        results = []
        for tracker in reversed(self.trackers):

            # Only output if:
            # 1. Recently updated (time_since_update < max_age)
            # 2. Confirmed (hit_streak >= min_hits OR early frames)
            if (tracker.time_since_update < self.max_age) and \
               (tracker.hit_streak >= self.min_hits or
                self.frame_count <= self.min_hits):

                box = tracker.get_state()[0]  # [x1,y1,x2,y2]
                results.append(
                    [*box, tracker.id + 1]    # +1 so IDs start at 1
                )

        # Delete dead trackers
        self.trackers = [
            t for t in self.trackers
            if t.time_since_update <= self.max_age
        ]

        if len(results) > 0:
            return np.array(results)
        else:
            return np.empty((0, 5))

Writing sort_object_tracking/sort.py


In [9]:
%%writefile sort_object_tracking/detector.py
import torch
import cv2
import numpy as np


class YOLOv3Detector:
    """
    YOLOv3 detector using PyTorch Hub.
    Returns detections as [x1, y1, x2, y2, confidence]
    """
    def __init__(self, conf_threshold=0.5, classes=None):
        """
        conf_threshold: minimum confidence to keep detection
        classes:        list of class names to track (None = all)
                        e.g. ['person', 'car', 'truck']
        """
        print("Loading YOLOv3...")
        self.model = torch.hub.load(
            'ultralytics/yolov3',
            'yolov3',
            pretrained=True
        )
        self.model.eval()
        self.conf_threshold = conf_threshold
        self.classes = classes
        print("YOLOv3 loaded!")

    def detect(self, frame):
        """
        Run detection on one frame.

        frame: BGR image (from cv2.imread or VideoCapture)
        Returns: numpy array (N, 5) [x1,y1,x2,y2,confidence]
        """
        # YOLOv3 expects RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Run inference
        results = self.model(rgb_frame)

        # Parse results
        detections = []
        for *box, conf, cls_id in results.xyxy[0].cpu().numpy():
            # Filter by confidence
            if conf < self.conf_threshold:
                continue

            # Filter by class if specified
            if self.classes is not None:
                class_name = results.names[int(cls_id)]
                if class_name not in self.classes:
                    continue

            x1, y1, x2, y2 = box
            detections.append([x1, y1, x2, y2, conf])

        if len(detections) == 0:
            return np.empty((0, 5))

        return np.array(detections, dtype=np.float32)

Writing sort_object_tracking/detector.py


In [19]:
%%writefile sort_object_tracking/main.py
import cv2
import numpy as np
from sort import SORT
from detector import YOLOv3Detector

# ── Colors for visualization ──────────────────────────────
np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(200, 3))

def draw_tracks(frame, tracks):
    """Draw bounding boxes and IDs on frame."""
    for track in tracks:
        x1, y1, x2, y2, track_id = track
        x1,y1,x2,y2 = int(x1),int(y1),int(x2),int(y2)
        track_id = int(track_id)

        # Unique color per ID
        color = COLORS[track_id % len(COLORS)].tolist()

        # Draw box
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

        # Draw ID label
        label = f"ID: {track_id}"
        cv2.putText(
            frame, label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, color, 2
        )
    return frame


def run_tracking(video_path, output_path=None):
    # ── Initialize ────────────────────────────────────────
    detector = YOLOv3Detector(
        conf_threshold=0.5,
        classes=['person', 'car', 'truck', 'bus']
    )

    tracker = SORT(
        max_age=1,        # from paper
        min_hits=3,       # from paper
        iou_threshold=0.3 # from paper
    )

    cap = cv2.VideoCapture(video_path)

    # Output video writer
    if output_path:
        fps    = int(cap.get(cv2.CAP_PROP_FPS))
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps, (width, height)
        )

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        print(f"Frame {frame_idx}")

        # ── Step 1: Detect ────────────────────────────────
        detections = detector.detect(frame)
        # detections: (N, 5) [x1,y1,x2,y2,conf]

        # ── Step 2: Track ─────────────────────────────────
        tracks = tracker.update(detections)
        # tracks: (M, 5) [x1,y1,x2,y2,track_id]

        # ── Step 3: Visualize ─────────────────────────────
        frame = draw_tracks(frame, tracks)

        # Show detection count and track count
        cv2.putText(
            frame,
            f"Detections: {len(detections)}  Tracks: {len(tracks)}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1, (255,255,255), 2
        )

        cv2.imshow("SORT Tracking", frame)

        if output_path:
            writer.write(frame)

        # Press q to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    if output_path:
        writer.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    for video_path in input_folder.glob("*.mp4"):
        output_path = output_folder / video_path.name
    
        run_tracking(
            video_path=str(video_path),
            output_path=str(output_path),
            model_path=None
        )
##remember to do this at the start 
#pip install filterpy scipy numpy opencv-python torch torchvision if required especially the opencv others are inbuilt in

Overwriting sort_object_tracking/main.py


In [ ]:
"""
Frame arrives
     ↓
YOLOv3 → detections (N, 5) [x1,y1,x2,y2,conf]
     ↓
SORT.update(detections):
     ↓
  1. Each KalmanBoxTracker.predict()
     → Kalman PREDICT step
     → estimated position this frame
     ↓
  2. associate_detections_to_trackers()
     → build IoU matrix (N detections × M trackers)
     → Hungarian algorithm finds optimal assignment
     → filter by iou_threshold=0.3
     ↓
  3. Matched pairs → KalmanBoxTracker.update(det)
     → Kalman UPDATE step
     → refine estimate using real detection
     ↓
  4. Unmatched detections → new KalmanBoxTracker
     → new object appeared
     ↓
  5. Unmatched trackers → time_since_update++
     → object disappeared or occluded
     → delete if time_since_update > max_age
     ↓
Output: (M, 5) [x1,y1,x2,y2,track_id]
     ↓
Draw boxes + IDs on frame
"""